# Make Your Python Code 20x Faster
## Threads vs Processes Explained
David Oluyale | Agentic AI Hub

Video 1 of **Systems Programming Explained: Advanced Programming Concepts Made Easy**.

We learn threads, the GIL, processes, races, locks, deadlocks, asyncio and parallel agent tool calls using everyday analogies first. All code is Python standard library; no API keys, sign-ins or package installs needed. Downloads and tools are deliberately simulated, using timed waits on your own machine.

Run cells from top to bottom. Timing depends on your machine and background load. The video used CPython 3.13.5 on an 8-logical-CPU Linux machine, standard GIL enabled. The GIL discussion applies to standard GIL-enabled CPython; optional free-threaded builds and native libraries can behave differently.

Code: https://github.com/davolu/systems-programming-explained/tree/main/01-threads-vs-processes

### 1. Waiting work: a pretend download

Imagine a cook waiting for a delivery. `time.sleep(1)` pauses for one second, like waiting on the network. These downloads are simulated locally: no real website is contacted. `download(n)` returns a message for file number `n`.

In [1]:
import time

def download(n):
    time.sleep(1)  # pretend: wait 1 second for the network
    return f"file {n} done"

### One cook, one at a time

Start a stopwatch with `time.perf_counter()`. Count from 1 to 20 (`range` stops before 21), run each download, and print it. Subtract the start time from the current time to measure elapsed seconds; `:.1f` rounds to one decimal place.

In [2]:
start = time.perf_counter()

for n in range(1, 21):
    print(download(n))

print(f"One cook: {time.perf_counter() - start:.1f} seconds")

file 1 done
file 2 done
file 3 done
file 4 done
file 5 done
file 6 done
file 7 done
file 8 done
file 9 done
file 10 done
file 11 done
file 12 done
file 13 done
file 14 done
file 15 done
file 16 done
file 17 done
file 18 done
file 19 done
file 20 done
One cook: 20.0 seconds


### Many cooks: ThreadPoolExecutor

`ThreadPoolExecutor` manages a team of threads (cooks sharing one kitchen). `max_workers=20` permits up to 20 workers. `pool.map` sends every file number to the function; `list` waits and collects results in input order. The `with` block tidies up the workers. Threads overlap the waits; they do not make a real network unlimited.

In [3]:
from concurrent.futures import ThreadPoolExecutor

start = time.perf_counter()

with ThreadPoolExecutor(max_workers=20) as pool:
    results = list(pool.map(download, range(1, 21)))

print(results[:3], "...")
print(f"Many cooks: {time.perf_counter() - start:.1f} seconds")

['file 1 done', 'file 2 done', 'file 3 done'] ...
Many cooks: 1.0 seconds


### 2. Thinking work: chopping vegetables

This cook never waits. The function adds the square of every number from zero up to, but not including, `n`. `jobs` contains eight copies of ten million. We will run exactly the same jobs three ways.

In [4]:
def chop(n):
    total = 0
    for i in range(n):
        total += i * i  # pure thinking work, no waiting
    return total

jobs = [10_000_000] * 8  # 8 big jobs

### Does adding cooks help thinking work?

Time the eight jobs one by one, then with eight threads. In the usual GIL-enabled CPython, only one thread at a time executes Python bytecode (the small instructions Python runs). Threads still take turns doing this pure-Python calculation. This is different from libraries such as NumPy that can release the GIL.

In [5]:
start = time.perf_counter()
for n in jobs:
    chop(n)
print(f"One cook:    {time.perf_counter() - start:.1f} s")

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as pool:
    list(pool.map(chop, jobs))
print(f"8 threads:   {time.perf_counter() - start:.1f} s")

One cook:    2.3 s
8 threads:   2.5 s


### More kitchens: ProcessPoolExecutor

A process has its own Python interpreter and memory: a separate kitchen and rule book (GIL). A CPU core is one worker inside the chip. More processes can calculate on different cores. `os.cpu_count()` reports logical CPUs; available CPU quota can be smaller. Process startup and passing data cost time. Eight workers helped on the recorded machine; your result will differ.

**Colab note:** on Linux-based Colab the function defined above commonly works with the default process start method. On newer Python or Windows/macOS, pool functions must live in an importable module. If this cell fails, use the script command below instead. The companion script correctly uses `if __name__ == "__main__":`.

If the pool cell does not run in your notebook, download `03_gil_threads_vs_processes.py` from the code folder, then run it in a separate process with:
```python
!python 03_gil_threads_vs_processes.py
```
This also works when functions defined inside the notebook cannot be imported by child processes.

In [6]:
import os
from concurrent.futures import ProcessPoolExecutor

print("CPU cores on this machine:", os.cpu_count())

start = time.perf_counter()
with ProcessPoolExecutor(max_workers=8) as pool:
    list(pool.map(chop, jobs))
print(f"8 processes: {time.perf_counter() - start:.1f} s")

CPU cores on this machine: 8
8 processes: 0.4 s


### 3. Race condition: two cooks update the same tip jar

A race condition means the result depends on timing. A deposit has three steps: read the jar, get distracted, write back the old total plus one. `global` lets the function change the shared name. `time.sleep(0)` deliberately yields execution, making the bug obvious. `_` is an unused loop variable. Make five threads, start them, and `join` (wait for) all of them. Five times one thousand should be five thousand. The wrong result varies each run; the video measured 1006. The GIL does not make a whole read-modify-write sequence atomic (indivisible).

In [7]:
import threading

balance = 0

def deposit():
    global balance
    for _ in range(1000):
        current = balance      # 1. read the jar
        time.sleep(0)          # 2. get distracted for a moment
        balance = current + 1  # 3. write the new total

threads = [threading.Thread(target=deposit) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()
print("Expected 5000, got", balance)

Expected 5000, got 1006


### The bathroom key: Lock

Only the holder of the key can enter. `with lock:` acquires the lock, runs the indented block, then releases it even on an error. Protect the complete read-pause-write sequence, not just the final assignment. Keep the protected part short, because other workers wait.

In [8]:
lock = threading.Lock()
balance = 0

def safe_deposit():
    global balance
    for _ in range(1000):
        with lock:  # only one thread inside at a time
            current = balance
            time.sleep(0)
            balance = current + 1

threads = [threading.Thread(target=safe_deposit) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()
print("Expected 5000, got", balance)

Expected 5000, got 5000


### 4. Deadlock: each cook has the tool the other needs

Cook A holds the knife and waits for the board; Cook B holds the board and waits for the knife. `with first` holds the first lock. `second.acquire(timeout=2)` tries the second lock for at most two seconds; `release()` puts it back. This timeout makes the demonstration safe rather than hanging forever. The star in `*a_tools` unpacks a pair into two arguments.

In [9]:
knife, board = threading.Lock(), threading.Lock()

def cook(name, first, second):
    with first:                        # grab the first tool
        time.sleep(0.1)
        if second.acquire(timeout=2):  # wait at most 2 s for the other
            second.release()
            print(name, "finished")
        else:
            print(name, "is stuck: deadlock!")

def kitchen(a_tools, b_tools):
    a = threading.Thread(target=cook, args=("Cook A", *a_tools))
    b = threading.Thread(target=cook, args=("Cook B", *b_tools))
    a.start(); b.start(); a.join(); b.join()

### Opposite order, then same order

Both cooks need the same two tools. First give them opposite orders, then the same order. With opposite orders, at least one cook times out before the other can finish. With the same order, one waits without holding the other tool, so both finish. Timing and which cook times out can vary.

In [10]:
print("Opposite order:")
kitchen((knife, board), (board, knife))

print("Same order:")
kitchen((knife, board), (knife, board))

Opposite order:
Cook A is stuck: deadlock!
Cook B finished
Same order:
Cook A finished
Cook B finished


### 5. asyncio: one organised waiter

One waiter takes orders and serves other tables while food cooks. `async def` creates a function that can pause cooperatively. `await asyncio.sleep(1)` yields control while waiting; do not replace it with blocking `time.sleep`. The list contains coroutine objects (jobs prepared to run); `gather(*tasks)` schedules all of them and waits for their results. In a notebook use `await main()`; in a script use `asyncio.run(main())`. This overlaps waits, not Python calculations.

In [11]:
import asyncio

async def download_async(n):
    await asyncio.sleep(1)  # wait, but let others work meanwhile
    return f"file {n} done"

async def main():
    start = time.perf_counter()
    tasks = [download_async(n) for n in range(1, 21)]
    results = await asyncio.gather(*tasks)
    print(len(results), "files in", f"{time.perf_counter() - start:.1f} s, on 1 thread")

await main()

20 files in 1.0 s, on 1 thread


### 6. AI agents: independent tools together

A waiter can ask for weather, calendar and search together if none needs the result of another. These three tools are simulated delays, not live services or real calendar access. The dictionary maps names to durations. `.items()` gives each name-duration pair. `if parallel` chooses `gather` rather than the sequential loop; the generator expression prepares one call per tool. `*` unpacks those calls for `gather`. The printed time is measured, not predicted.

Real agents must respect tool permissions, rate limits and dependencies. Do not parallelise a payment and its approval, or a booking that depends on the search result.

In [12]:
async def call_tool(name, seconds):
    await asyncio.sleep(seconds)  # pretend the tool is busy
    return f"{name} ready"

tools = {"weather": 1.0, "calendar": 1.5, "web_search": 2.0}

async def agent(parallel):
    start = time.perf_counter()
    if parallel:
        await asyncio.gather(*(call_tool(n, s) for n, s in tools.items()))
    else:
        for n, s in tools.items():
            await call_tool(n, s)
    print("Parallel:  " if parallel else "One by one:", f"{time.perf_counter() - start:.1f} s")

await agent(parallel=False)
await agent(parallel=True)

One by one: 4.5 s
Parallel:   2.0 s


## Recap

| Everyday picture | Python tool | Best fit |
|---|---|---|
| Many cooks, one kitchen | Threads | Waiting for network/files |
| Many kitchens | Processes | Pure-Python calculations on several cores |
| One bathroom key | Lock | Protecting shared state |
| One organised waiter | asyncio | Many cooperative waits |

Concurrency means overlapping progress; parallelism means simultaneous execution. Never assume a speed-up; time your own workload. **Next: Lock-Free Code Explained.**

Channel: https://www.youtube.com/@AgenticAIHub1
Python Crash Course: https://youtu.be/ZaY_l07IZSo